# Fig5: 24 h d05 10 m wind snapshots

Generate comparable 10 m wind fields for Ragasa at 2025-09-23 22:00 UTC and Yagi at 2024-09-05 12:00 UTC. All five models and both storms share one wind-speed colour scale.

In [1]:
from __future__ import annotations

from pathlib import Path

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr


MODELS = ("pangu", "graphcast", "fengwu", "fuxi", "aurora")
CASES = (
    {
        "name": "ragasa",
        "time": "2025-09-23_22:00:00",
        "files": {model: f"uv10_d05_{model}_24h.nc" for model in MODELS},
    },
    {
        "name": "yagi",
        "time": "2024-09-05_12:00:00",
        "files": {
            model: f"24h_{model}_uv10_d05_2024-09-05_12UTC.nc"
            for model in MODELS
        },
    },
)

PLOT_CMAP = "coolwarm"
QUIVER_STEP = 10
QUIVER_SCALE = 500
QUIVER_WIDTH = 0.0035
QUIVER_HEADWIDTH = 3.0
QUIVER_COLOR = "black"
FIGSIZE = (6, 6)
DPI = 600
LEGEND_FONTSIZE = 16

plt.rcParams["font.family"] = "Arial"
plt.rcParams["font.size"] = 12


In [2]:
def _find_fig5_dir() -> Path:
    """Locate Fig5 when the notebook is run from Fig5 or the workspace root."""
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd / "Fig5", cwd / "Figs" / "Fig5"]
    candidates.extend(parent / "Fig5" for parent in cwd.parents)

    seen: set[Path] = set()
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        if (candidate / "input" / "ragasa").is_dir() and (candidate / "input" / "yagi").is_dir():
            return candidate

    raise FileNotFoundError(
        "Could not locate Fig5/input/{ragasa,yagi} from "
        f"the current working directory: {cwd}"
    )


def _time_tag(time_str: str) -> str:
    date_part, clock_part = time_str.split("_")
    return f"{date_part.replace('-', '')}_{clock_part.replace(':', '')}"


def _as_text(value) -> str:
    return value.decode() if isinstance(value, (bytes, np.bytes_)) else str(value)


def _times_to_strings(times_da: xr.DataArray) -> np.ndarray:
    values = times_da.values
    if times_da.ndim == 1:
        return np.array([_as_text(value) for value in values])
    if times_da.ndim == 2:
        return np.array(["".join(_as_text(value) for value in row) for row in values])
    raise ValueError(
        f"Unsupported TIMES dimensions {times_da.dims} with shape {times_da.shape}"
    )


def _select_exact_time(ds: xr.Dataset, target_time: str, path: Path) -> xr.Dataset:
    times_da = ds["TIMES"]
    time_dim = times_da.dims[0]
    available = _times_to_strings(times_da)
    matches = np.flatnonzero(available == target_time)
    if len(matches) != 1:
        raise KeyError(
            f"Expected exactly one {target_time!r} record in {path}; "
            f"found {len(matches)}. Available TIMES: {available.tolist()}"
        )
    return ds.isel({time_dim: int(matches[0])})


def _load_snapshot(path: Path, target_time: str) -> dict[str, np.ndarray]:
    if not path.is_file():
        raise FileNotFoundError(f"Missing 24h/d05 input file: {path}")

    with xr.open_dataset(path) as ds:
        required = {"U10", "V10", "LAT", "LON"}
        missing = sorted(required - set(ds.variables))
        if missing:
            raise KeyError(f"Missing variable(s) {missing} in {path}")

        if "TIMES" in ds.variables:
            snapshot = _select_exact_time(ds, target_time, path)
        else:
            u_ndim = ds["U10"].squeeze().ndim
            v_ndim = ds["V10"].squeeze().ndim
            if u_ndim != 2 or v_ndim != 2:
                raise KeyError(
                    f"{path} has no TIMES variable and is not a single 2-D snapshot: "
                    f"U10 ndim={u_ndim}, V10 ndim={v_ndim}"
                )
            snapshot = ds

        arrays = {
            name: np.asarray(snapshot[name].squeeze().values).copy()
            for name in required
        }

    shapes = {name: array.shape for name, array in arrays.items()}
    bad_dims = {name: shape for name, shape in shapes.items() if len(shape) != 2}
    if bad_dims:
        raise ValueError(f"Expected 2-D wind/grid variables in {path}; got {bad_dims}")
    if len(set(shapes.values())) != 1:
        raise ValueError(f"U10/V10/LAT/LON shapes differ in {path}: {shapes}")
    if not np.isfinite(arrays["LAT"]).any() or not np.isfinite(arrays["LON"]).any():
        raise ValueError(f"LAT/LON contain no finite coordinates in {path}")

    return arrays


def _plot_one_map(snapshot: dict[str, np.ndarray], norm, cmap):
    u10 = snapshot["U10"]
    v10 = snapshot["V10"]
    lat = snapshot["LAT"]
    lon = snapshot["LON"]
    speed = np.hypot(u10, v10)

    fig = plt.figure(figsize=FIGSIZE)
    fig.subplots_adjust(right=0.88)
    ax = plt.axes(projection=ccrs.PlateCarree())
    ax.set_extent(
        [float(np.nanmin(lon)), float(np.nanmax(lon)),
         float(np.nanmin(lat)), float(np.nanmax(lat))],
        crs=ccrs.PlateCarree(),
    )
    ax.pcolormesh(
        lon, lat, speed, cmap=cmap, norm=norm, shading="auto",
        transform=ccrs.PlateCarree(),
    )
    ax.add_feature(cfeature.COASTLINE.with_scale("10m"), linewidth=0.7)
    ax.quiver(
        lon[::QUIVER_STEP, ::QUIVER_STEP],
        lat[::QUIVER_STEP, ::QUIVER_STEP],
        u10[::QUIVER_STEP, ::QUIVER_STEP],
        v10[::QUIVER_STEP, ::QUIVER_STEP],
        color=QUIVER_COLOR,
        scale=QUIVER_SCALE,
        width=QUIVER_WIDTH,
        headwidth=QUIVER_HEADWIDTH,
        transform=ccrs.PlateCarree(),
    )
    return fig


def _export_horizontal_colorbar(norm, cmap, out_path: Path) -> None:
    fig = plt.figure(figsize=(9, 0.9))
    cax = fig.add_axes([0.06, 0.40, 0.88, 0.22])
    scalar_mappable = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
    scalar_mappable.set_array([])
    colorbar = fig.colorbar(scalar_mappable, cax=cax, orientation="horizontal")
    colorbar.set_label("10 m wind speed (m s$^{-1}$)", fontsize=LEGEND_FONTSIZE)
    colorbar.ax.tick_params(labelsize=LEGEND_FONTSIZE)
    fig.savefig(out_path, dpi=DPI, bbox_inches="tight")
    plt.close(fig)


In [3]:
fig5_dir = _find_fig5_dir()
input_dir = fig5_dir / "input"
output_dir = fig5_dir / "output"
output_dir.mkdir(parents=True, exist_ok=True)

snapshots: dict[tuple[str, str], dict[str, np.ndarray]] = {}
for case in CASES:
    for model in MODELS:
        path = input_dir / case["name"] / case["files"][model]
        snapshots[(case["name"], model)] = _load_snapshot(path, case["time"])
        print(f"Loaded {case['name']} {model}: {path.name}")

finite_minima = []
finite_maxima = []
for snapshot in snapshots.values():
    speed = np.hypot(snapshot["U10"], snapshot["V10"])
    if np.isfinite(speed).any():
        finite_minima.append(float(np.nanmin(speed)))
        finite_maxima.append(float(np.nanmax(speed)))

if not finite_minima:
    raise ValueError("The 10 requested snapshots contain no finite wind speeds")

global_vmin = min(finite_minima)
global_vmax = max(finite_maxima)
if not global_vmax > global_vmin:
    raise ValueError(
        f"Invalid shared wind-speed range: {global_vmin} to {global_vmax} m/s"
    )

norm = mpl.colors.Normalize(vmin=global_vmin, vmax=global_vmax)
cmap = mpl.colormaps[PLOT_CMAP]
print(f"Shared colour range: {global_vmin:.3f} to {global_vmax:.3f} m/s")

generated_paths: list[Path] = []
for case in CASES:
    for model in MODELS:
        fig = _plot_one_map(snapshots[(case["name"], model)], norm=norm, cmap=cmap)
        out_path = output_dir / f"wind10_{model}_{_time_tag(case['time'])}.tif"
        fig.savefig(out_path, dpi=DPI, bbox_inches="tight")
        plt.close(fig)
        generated_paths.append(out_path)
        print(f"Saved: {out_path}")

colorbar_path = output_dir / "colorbar_wind10.tif"
_export_horizontal_colorbar(norm=norm, cmap=cmap, out_path=colorbar_path)
generated_paths.append(colorbar_path)

if len(generated_paths) != 11 or any(not path.is_file() or path.stat().st_size == 0 for path in generated_paths):
    raise RuntimeError("Expected 10 non-empty wind maps and one non-empty shared colour bar")

print(f"Done: generated 10 wind maps and one shared colour bar in {output_dir}")


Loaded ragasa pangu: uv10_d05_pangu_24h.nc
Loaded ragasa graphcast: uv10_d05_graphcast_24h.nc
Loaded ragasa fengwu: uv10_d05_fengwu_24h.nc
Loaded ragasa fuxi: uv10_d05_fuxi_24h.nc
Loaded ragasa aurora: uv10_d05_aurora_24h.nc
Loaded yagi pangu: 24h_pangu_uv10_d05_2024-09-05_12UTC.nc
Loaded yagi graphcast: 24h_graphcast_uv10_d05_2024-09-05_12UTC.nc
Loaded yagi fengwu: 24h_fengwu_uv10_d05_2024-09-05_12UTC.nc
Loaded yagi fuxi: 24h_fuxi_uv10_d05_2024-09-05_12UTC.nc
Loaded yagi aurora: 24h_aurora_uv10_d05_2024-09-05_12UTC.nc
Shared colour range: 0.258 to 60.845 m/s


Saved: E:\BaiduSyncdisk\Code\06_AI_WRF_UCM\Figs\Fig5\output\wind10_pangu_20250923_220000.tif


Saved: E:\BaiduSyncdisk\Code\06_AI_WRF_UCM\Figs\Fig5\output\wind10_graphcast_20250923_220000.tif


Saved: E:\BaiduSyncdisk\Code\06_AI_WRF_UCM\Figs\Fig5\output\wind10_fengwu_20250923_220000.tif


Saved: E:\BaiduSyncdisk\Code\06_AI_WRF_UCM\Figs\Fig5\output\wind10_fuxi_20250923_220000.tif


...

Saved: E:\BaiduSyncdisk\Code\06_AI_WRF_UCM\Figs\Fig5\output\wind10_graphcast_20240905_120000.tif


Saved: E:\BaiduSyncdisk\Code\06_AI_WRF_UCM\Figs\Fig5\output\wind10_fengwu_20240905_120000.tif


Saved: E:\BaiduSyncdisk\Code\06_AI_WRF_UCM\Figs\Fig5\output\wind10_fuxi_20240905_120000.tif


Saved: E:\BaiduSyncdisk\Code\06_AI_WRF_UCM\Figs\Fig5\output\wind10_aurora_20240905_120000.tif


Done: generated 10 wind maps and one shared colour bar in E:\BaiduSyncdisk\Code\06_AI_WRF_UCM\Figs\Fig5\output
